# Jayhind LLM Invoice-OCR — Kaggle / Colab launcher

**Kaggle:** notebook settings → Accelerator = **GPU T4 x2**, Internet = **On**.
**Colab:** Runtime → Change runtime type → **GPU**.

Run the cells top to bottom. The last cell prints `PUBLIC_URL` + `API_KEY` — copy them into `jayhind-admin-back/.env`, then `dev restart admin-back`, and keep this tab open.

> **Use ngrok, not the Cloudflare quick tunnel.** A `trycloudflare.com` tunnel drops any request that runs longer than ~100s (HTTP 524), and this GPU pipeline (PaddleOCR-VL + an extractor LLM) routinely takes longer per document. Paste your ngrok authtoken in **cell 2** so the tunnel has no such cap. Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken .

> Cell 1 is **safe to re-run** — it resets to a clean folder first, so it can never nest.

In [ ]:
# 1. get the code (reset-safe: always clones one clean copy)
import os
base = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.chdir(base)
!rm -rf jayhind-ocr-service
!git clone https://github.com/harshbpatel11/jayhind-ocr-service.git
os.chdir(f'{base}/jayhind-ocr-service')
print('working dir:', os.getcwd())

In [ ]:
# 2. ngrok authtoken (REQUIRED for the hosted GPU engine — no ~100s cap).
#    Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
#    Leave blank to fall back to a Cloudflare quick tunnel (will 524 on slow docs).
import os
os.environ['NGROK_TOKEN'] = ''  # <-- paste your ngrok authtoken between the quotes
assert os.environ['NGROK_TOKEN'].strip(), 'Set NGROK_TOKEN above (or delete this assert to use Cloudflare).'
print('ngrok token set:', bool(os.environ['NGROK_TOKEN'].strip()))

In [ ]:
# 3. install + run (first run downloads a few GB; leave it running)
!bash run.sh

### Test it (open a new cell while the one above keeps running)
```python
import requests
URL = 'https://xxxxx.ngrok-free.app'   # the PUBLIC_URL printed above
KEY = 'xxxx'                            # the API_KEY printed above
print(requests.get(f'{URL}/health', headers={'ngrok-skip-browser-warning':'true'}).json())
# r = requests.post(f'{URL}/parse',
#                   headers={'Authorization': f'Bearer {KEY}', 'ngrok-skip-browser-warning':'true'},
#                   files={'file': open('invoice.jpg','rb')}, timeout=300)
# print(r.json())
```